<a href="https://colab.research.google.com/github/idoschw3/Corpus2GeoRAG/blob/main/ApplyNER/NER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
from transformers import pipeline
from google.colab import userdata
import pandas as pd
import numpy as np
import subprocess
import json
import os

### Clone Corpus2GeoRAG Repository

In [62]:
%%bash
rm -rf Corpus2GeoRAG
git clone https://github.com/EtzionR/Corpus2GeoRAG.git

Cloning into 'Corpus2GeoRAG'...


### Data Loading and Initial Inspection

**Purpose:** To load the raw text data, which is crucial for subsequent NER tasks, from a JSON file. This section also includes an example to visually inspect the structure and content of a single entry, ensuring the data is correctly loaded and accessible.


**Input/Output:** Input is `/content/Corpus2GeoRAG/examples/DATA.json`.

**Date:** 2026-09-23

In [63]:
PATH = r'/content/Corpus2GeoRAG/examples/DATA.json'
PATH

'/content/Corpus2GeoRAG/examples/DATA.json'

In [64]:
with open(PATH, 'r') as file:
    data = json.load(file)

len(data)

897

In [65]:
[*data.keys()][:10]

['12 (song)',
 '14 May 2026 Russian strikes on Ukraine',
 '15th BRICS summit',
 '16 March 2022 Chernihiv breadline attack',
 '17 November 2024 Russian strikes on Ukraine',
 '18 March 2022 Mykolaiv military quarters attack',
 '1st European Political Community Summit',
 '2000 Meters to Andriivka',
 '2014 pro-Russian unrest in Ukraine',
 '2020s European re-armament']

In [66]:
example = np.random.choice([*data],1)[0]


print(f'PAGE: {example}:\n\n{data[example]}')


PAGE: 2023 Vilnius summit:

The 2023 Vilnius summit was the formal meeting of the heads of state and heads of government of the thirty-one members of the North Atlantic Treaty Organization (NATO), their partner countries, and the European Union, held in Vilnius, Lithuania, on 11–12 July 2023. The summit was officially proposed during the previous 2022 Madrid summit and its dates were fixed on 9 November 2022. It was notable for the discussions about the ongoing Russian invasion of Ukraine as well as Sweden and Ukraine's prospective memberships into the alliance.


== Background ==
The summit was held in the context of an ongoing Russian invasion of Ukraine. In his January 2023 address to the Lithuanian Parliament, President of Ukraine Volodymyr Zelenskyy described the summit as fateful. Ukraine expressed the desire to be formally invited to NATO at the Vilnius summit. On 8 July 2023, US President Joe Biden said that Ukraine is not ready to join NATO at that time. By the time of the sum

In [67]:
TITLE = '=='
MIN_LENGTH = 5

paragraphs = data[example].split('\n')
paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]

print(f'{len(paragraphs_processed)} Paragraphs Extracted from {len(paragraphs)}\n{round(len(paragraphs_processed)/len(paragraphs)*100,1)}%')

16 Paragraphs Extracted from 42
38.1%


### Text Preprocessing: Paragraph Filtering

**Purpose:** To perform an initial cleaning of the raw text by filtering out structural elements and short, uninformative lines. This step aims to prepare cleaner text segments for downstream NER processing by removing elements such as section titles and empty or minimal lines.

**Date:** 2026-09-24

In [68]:
TITLE = '=='
MIN_LENGTH = 5

data_processed = {}

total_paragraphs = 0
total_processed = 0

for key, text in data.items():
  paragraphs = text.split('\n')
  paragraphs_processed = [text for text in paragraphs if text.startswith(TITLE)==False and len(text)>=MIN_LENGTH]
  data_processed[key] = paragraphs_processed

  total_paragraphs += len(paragraphs)
  total_processed += len(paragraphs_processed)

print(f'{total_processed} paragraphs extracted from {total_paragraphs}\n {round(total_processed/total_paragraphs*100,1)}%')

42822 paragraphs extracted from 88029
 48.6%


In [69]:
example = np.random.choice([*data_processed],1)[0]

print(f'PAGE: {example}:\n\n{data_processed[example]}')

PAGE: Oleg Tsokov:

['Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Цоков; 23 September 1971 – 11 July 2023) was a Russian lieutenant general who served in the Russian Ground Forces as deputy commander of the Southern Military District. He was killed in 2023 by a missile strike during a Ukrainian counteroffensive against the Russian invasion of Ukraine.', "Oleg Yuryevich Tsokov was  the son of Yuri Georgievich, a military officer, and Alla Ivanovna, a physicist-mathematician. Since his family moved often, he grew up on military bases and studied at eleven different schools, including in Choibalsan and Semipalatinsk, before receiving his secondary education certificate. Following in his father's footsteps, Tsokov chose a military career and was accepted to the Tashkent Higher Combined Arms Command School. Graduating in 1994, he was posted to the 74th Separate Guards Motor Rifle Brigade at Yurga in the Siberian Military District as a motor rifle platoon commander. Tsokov was deployed to t

In [70]:
SAVE_PATH = '/content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json'

with open(SAVE_PATH, 'w', encoding='utf-8') as f:
    json.dump(data_processed, f, ensure_ascii=False, indent=2)

print(f'Saved to: {SAVE_PATH}')

Saved to: /content/Corpus2GeoRAG/ApplyNER/data_initial_cleaning.json


## Initial NER application

**Purpose:** To perform an initial NER application with base models and analyze the results

**Date:** 2026-09-24

### DistilBERT

In [71]:
MODEL_DistilBERT = "dslim/distilbert-NER"
MODEL_DistilBERT

ner_DistilBERT = pipeline("ner",
               model=MODEL_DistilBERT,
               aggregation_strategy='average')

ner_DistilBERT

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

TokenClassificationPipeline: {'model': 'DistilBertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [72]:
output_DistilBERT = ner_DistilBERT(data_processed[example])
output_DistilBERT

[[{'entity_group': 'PER',
   'score': np.float32(0.99566525),
   'word': 'Oleg Yuryevich Tsokov',
   'start': 0,
   'end': 21},
  {'entity_group': 'MISC',
   'score': np.float32(0.9325426),
   'word': 'Russian',
   'start': 23,
   'end': 30},
  {'entity_group': 'PER',
   'score': np.float32(0.8685282),
   'word': 'Олег',
   'start': 32,
   'end': 36},
  {'entity_group': 'PER',
   'score': np.float32(0.85478663),
   'word': 'Юрьевич Цоков',
   'start': 37,
   'end': 50},
  {'entity_group': 'MISC',
   'score': np.float32(0.99698824),
   'word': 'Russian',
   'start': 92,
   'end': 99},
  {'entity_group': 'ORG',
   'score': np.float32(0.97474045),
   'word': 'Russian Ground Forces',
   'start': 137,
   'end': 158},
  {'entity_group': 'ORG',
   'score': np.float32(0.8219959),
   'word': 'Southern Military District',
   'start': 186,
   'end': 212},
  {'entity_group': 'MISC',
   'score': np.float32(0.9975473),
   'word': 'Ukrainian',
   'start': 265,
   'end': 274},
  {'entity_group': 'MISC

In [78]:
df_DistilBERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_DistilBERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.995665,Oleg Yuryevich Tsokov,0,21
1,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,MISC,0.932543,Russian,23,30
2,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.868528,Олег,32,36
3,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.854787,Юрьевич Цоков,37,50
4,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,MISC,0.996988,Russian,92,99
...,...,...,...,...,...,...,...,...
91,Oleg Tsokov,11,Order of Military Merit,MISC,0.591978,Merit,18,23
92,Oleg Tsokov,12,Military operation in Syria medal,LOC,0.997561,Syria,22,27
93,Oleg Tsokov,13,List of Russian generals killed during the Rus...,MISC,0.995336,Russian,8,15
94,Oleg Tsokov,13,List of Russian generals killed during the Rus...,MISC,0.996570,Russian,43,50


### BERT

In [74]:
MODEL_BERT = "dslim/bert-base-NER"
MODEL_BERT

ner_BERT = pipeline("ner",
               model=MODEL_BERT,
               aggregation_strategy='average')

ner_BERT

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TokenClassificationPipeline: {'model': 'BertForTokenClassification', 'dtype': 'float32', 'device': 'cpu', 'input_modalities': 'text'}

In [75]:
output_BERT = ner_BERT(data_processed[example])
output_BERT

[[{'entity_group': 'PER',
   'score': np.float32(0.9833424),
   'word': 'Oleg Yuryevich Tsokov',
   'start': 0,
   'end': 21},
  {'entity_group': 'MISC',
   'score': np.float32(0.9988771),
   'word': 'Russian',
   'start': 23,
   'end': 30},
  {'entity_group': 'PER',
   'score': np.float32(0.9521918),
   'word': 'Олег Юрьевич',
   'start': 32,
   'end': 44},
  {'entity_group': 'MISC',
   'score': np.float32(0.9997689),
   'word': 'Russian',
   'start': 92,
   'end': 99},
  {'entity_group': 'ORG',
   'score': np.float32(0.9957163),
   'word': 'Russian Ground Forces',
   'start': 137,
   'end': 158},
  {'entity_group': 'LOC',
   'score': np.float32(0.4995426),
   'word': 'Southern',
   'start': 186,
   'end': 194},
  {'entity_group': 'ORG',
   'score': np.float32(0.8658221),
   'word': 'Military',
   'start': 195,
   'end': 203},
  {'entity_group': 'LOC',
   'score': np.float32(0.7493531),
   'word': 'District',
   'start': 204,
   'end': 212},
  {'entity_group': 'MISC',
   'score': np.f

In [79]:
df_BERT = pd.DataFrame([
    {
        "page": example,
        "paragraph_id": paragraph_id,
        "paragraph": data_processed[example][paragraph_id],
        **entity
    }
    for paragraph_id, entities in enumerate(output_BERT)
    for entity in entities
])

df_DistilBERT

,page,paragraph_id,paragraph,entity_group,score,word,start,end
0,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.995665,Oleg Yuryevich Tsokov,0,21
1,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,MISC,0.932543,Russian,23,30
2,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.868528,Олег,32,36
3,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,PER,0.854787,Юрьевич Цоков,37,50
4,Oleg Tsokov,0,Oleg Yuryevich Tsokov (Russian: Олег Юрьевич Ц...,MISC,0.996988,Russian,92,99
...,...,...,...,...,...,...,...,...
91,Oleg Tsokov,11,Order of Military Merit,MISC,0.591978,Merit,18,23
92,Oleg Tsokov,12,Military operation in Syria medal,LOC,0.997561,Syria,22,27
93,Oleg Tsokov,13,List of Russian generals killed during the Rus...,MISC,0.995336,Russian,8,15
94,Oleg Tsokov,13,List of Russian generals killed during the Rus...,MISC,0.996570,Russian,43,50


## Commit to repo

In [77]:
REPO_PATH = "/content/Corpus2GeoRAG"
GITHUB_USER = "idoschw3"
GITHUB_REPO = "Corpus2GeoRAG"
GITHUB_EMAIL = "193781032+idoschw3@users.noreply.github.com"
COMMIT_MESSAGE = "Update NER workflow"

# Move to repository
os.chdir(REPO_PATH)

# Set Git identity for this repository
subprocess.run(["git", "config", "user.name", GITHUB_USER], check=True)
subprocess.run(["git", "config", "user.email", GITHUB_EMAIL], check=True)

# Make sure origin points to my personal fork
subprocess.run([
    "git", "remote", "set-url", "origin",
    f"https://github.com/{GITHUB_USER}/{GITHUB_REPO}.git"
], check=True)

# Stage ALL new, modified, and deleted files
subprocess.run(["git", "add", "-A"], check=True)

# Commit only if there are changes
status = subprocess.run(
    ["git", "status", "--porcelain"],
    capture_output=True,
    text=True,
    check=True
)

if status.stdout.strip():
    subprocess.run(
        ["git", "commit", "-m", COMMIT_MESSAGE],
        check=True
    )
    print("Changes committed.")
else:
    print("No new changes to commit.")

# Read GitHub token securely from Colab Secrets
github_token = userdata.get("GITHUB_TOKEN")

# Temporary authentication helper
askpass_path = "/tmp/github_askpass.sh"

with open(askpass_path, "w") as f:
    f.write(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) echo "$GITHUB_USER" ;;\n'
        '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
        'esac\n'
    )

os.chmod(askpass_path, 0o700)

env = os.environ.copy()
env["GITHUB_USER"] = GITHUB_USER
env["GITHUB_TOKEN"] = github_token
env["GIT_ASKPASS"] = askpass_path
env["GIT_TERMINAL_PROMPT"] = "0"

# Push the current branch to my fork
branch = subprocess.run(
    ["git", "branch", "--show-current"],
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

try:
    subprocess.run(
        ["git", "push", "origin", branch],
        env=env,
        check=True
    )
    print(f"Successfully pushed to {GITHUB_USER}/{GITHUB_REPO} on branch '{branch}'.")
finally:
    if os.path.exists(askpass_path):
        os.remove(askpass_path)

No new changes to commit.
Successfully pushed to idoschw3/Corpus2GeoRAG on branch 'main'.
